In [9]:
%pip install psycopg2
%pip install pandas
%pip install requests
%pip install truststore

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import psycopg2
import psycopg2.extras
import pandas as pd
import requests
import json
import time
from datetime import datetime
import csv
import os.path
from datetime import date, timedelta

# [note 7] www.tmd.go.th ส่ง certificate มาไม่ครบ (ขาด intermediate CA ของ GlobalSign)
#           -> requests ขึ้น SSLError: CERTIFICATE_VERIFY_FAILED ทุก request = สาเหตุที่รันไม่ได้
#           ใช้ truststore ให้ Python ตรวจ certificate ผ่าน Windows (เหมือน browser) ซึ่งไปดึง intermediate ที่ขาดมาเองได้
#           (ยังตรวจ SSL ตามปกติ ไม่ใช่การปิดด้วย verify=False)
import truststore   # new

truststore.inject_into_ssl()   # new

In [12]:
# use for load station data

# api_url = "http://localhost/tmd/station.json"   # old
api_url = "https://data.tmd.go.th/api/Station/v1/?uid=api&ukey=api12345&format=json"   # new

# station_file = "D:/PhD-Semester-2/Code-Download/loadTMDStation_clean/tmd_station.csv"   # old
station_file = "tmd_station.csv"   # new
out_dir = "tmd_daily"   # new
os.makedirs(out_dir, exist_ok=True)   # new

In [13]:

def text_or_blank(v):
    return v if isinstance(v, str) else ""

if (not os.path.isfile(station_file)):
    response = requests.get(api_url, timeout=60)
    print("response status: %d" % response.status_code)
    if (response.status_code == 200):
        data_list = response.json()["Station"]
        with open(station_file, 'w', encoding="tis-620", errors="replace", newline='') as file:
            writer = csv.writer(file)
            header_row = ["station_id","station_name_th","station_name_en","province","lat","lon"]
            writer.writerow(header_row)
            for dd in data_list:
                drow = [text_or_blank(dd["WmoCode"]),text_or_blank(dd["StationNameThai"]),text_or_blank(dd["StationNameEnglish"]),text_or_blank(dd["Province"]),text_or_blank(dd["Latitude"]),text_or_blank(dd["Longitude"])]
                writer.writerow(drow)
        print("saved:", station_file, len(data_list), "stations")
else:
    print("use existing:", station_file)

use existing: tmd_station.csv


In [ ]:
# [note 2] range() ไม่รวมตัวสุดท้าย 
def daterange(start_date, end_date):
    # for n in range(int((end_date - start_date).days)):   # old
    for n in range(int((end_date - start_date).days) + 1):   # new
        yield start_date + timedelta(n)

In [ ]:
#tmd data for daily

#daily_api_url = "http://localhost/tmd/rain_daily.json?DateTime=#date%2017:00&Culture=th-TH"
daily_api_url = "https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=#date%2017:00&Culture=th-TH"
# [note 10] station_id บางสถานีว่าง -> pandas อ่านทั้งคอลัมน์เป็น float -> ได้ 48380.0 แทน 48380 -> บังคับให้อ่านเป็นข้อความ
# df = pd.read_csv(station_file, encoding = 'tis-620')   # old
df = pd.read_csv(station_file, encoding = 'tis-620', dtype={'station_id': str})   # new
sel_df = df.loc[df['station_name_th'] == 'สมุทรปราการ สกษ.'].iloc[0]
print(sel_df['station_id'],sel_df['station_name_th'],sel_df['station_name_en'],sel_df['lat'],sel_df['lon'])

#specify date
start_date = date(2026, 1, 1)
end_date = date(2026, 12, 31)
unmatched = set()   # [note 6] new: เก็บชื่อสถานีที่หาใน tmd_station.csv ไม่เจอ
for single_date in daterange(start_date, end_date):
    cur_date = single_date.strftime("%Y-%m-%d")
    print(single_date.strftime("%Y-%m-%d"))
    api = daily_api_url.replace("#date",cur_date)
    print(api)

    # response = requests.get(api)   # old
    try:   # new
        response = requests.get(api, timeout=60)
    except requests.exceptions.RequestException as e:
        # [note 12] เดิม continue -> ข้ามวันที่ดึงไม่ได้ไปเงียบ ๆ แล้ว cell/Airflow task ยังขึ้นว่าสำเร็จ
        #           ใหม่: พิมพ์ข้อความแล้ว raise ต่อ -> task fail -> Airflow retry หรือแจ้ง failed
        print("request failed:", cur_date, e)
        # continue   # old
        raise   # new
    if (response.status_code == 200):
        response_json =response.json()
        data_list = response_json["items"]
        data_size = len(data_list)

        # data_file = "D:/PhD-Semester-2/Code-Download/loadTMDStation_clean/tmd_daily_all_2024/tmd_"+cur_date+".csv"   # old
        data_file = out_dir + "/tmd_" + cur_date + ".csv"   # new


        # with open(data_file, 'a', encoding="tis-620", newline='') as file:   # old 
        with open(data_file, 'w', encoding="tis-620", errors="replace", newline='') as file:   # new
            writer = csv.writer(file)
            header_row = ["station_id","station_name_th","station_name_en","province","lat","lon","date","rain(mm)","maxTemp(c)","minTemp(c)","maxWindDirection","maxWindSpeed(km/h)"]
            writer.writerow(header_row)
            for dd in data_list:
                # [note 6] เดิม .iloc[0] บนผลลัพธ์ว่าง (ชื่อจาก API ไม่ตรงกับ csv) -> IndexError หยุดทั้ง loop
                #           เช็กก่อน ถ้าไม่เจอ: ยังเขียนข้อมูลอากาศไว้ (ใช้ stationId/ชื่อจาก API, ช่องอื่นว่าง) และจดชื่อไว้ใน unmatched
                # sel_df = df.loc[df['station_name_th'] == dd["stationName"]].iloc[0]   # old
                match = df.loc[df['station_name_th'] == dd["stationName"]]   # new
                if (match.empty):
                    unmatched.add(dd["stationName"])
                    drow = [dd["stationId"],dd["stationName"],"","","","",cur_date,dd["rainfall"],dd["maxTemperature"],dd["minTemperature"],dd["maxWindDirection"],dd["maxWindSpeed"]]
                else:
                    sel_df = match.iloc[0]
                    drow = [sel_df['station_id'],sel_df['station_name_th'],sel_df['station_name_en'],sel_df['province'],sel_df['lat'],sel_df['lon'],cur_date,dd["rainfall"],dd["maxTemperature"],dd["minTemperature"],dd["maxWindDirection"],dd["maxWindSpeed"]]
                writer.writerow(drow)
    else:
        print("request failed:", cur_date, "status", response.status_code)
        # [note 12] status ไม่ใช่ 200 ก็ต้อง fail เหมือนกัน ไม่ใช่แค่ print
        raise RuntimeError("TMD daily API returned %d for %s" % (response.status_code, cur_date))   # new

print("stations not found in", station_file, ":", len(unmatched))   # [note 6] new
for name in sorted(unmatched):
    print("  ", name)

48420 สมุทรปราการ สกษ. Samut Prakarn 13.516666666667 100.76166666667
2024-01-01
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-01%2017:00&Culture=th-TH
2024-01-02
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-02%2017:00&Culture=th-TH
2024-01-03
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-03%2017:00&Culture=th-TH
2024-01-04
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-04%2017:00&Culture=th-TH
2024-01-05
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-05%2017:00&Culture=th-TH
2024-01-06
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-06%2017:00&Culture=th-TH
2024-01-07
https://www.tmd.go.th/api/WeatherObservationDailyStation/weather-station-daily?DateTime=2024-01-07%2017:00&Culture=th-TH
2024-01

KeyboardInterrupt: 

In [ ]:

#tmd data for today

# [note 9] new: ดึงฝน "วันนี้" ของทุกสถานี TMD (ทำแบบเดียวกับ [note 6] ใน 02_Station_Thaiwater)
#   - API รายวัน (weather-station-daily) ออกช้า ~2 วัน -> ใช้ดึง "วันนี้" ไม่ได้
#     จึงใช้ Weather3Hours ซึ่งอัปเดตทุก 3 ชม. (1:00, 4:00, 7:00, ... 22:00) แทน
#   - rain(mm) = Rainfall24Hr = ฝนสะสม 24 ชม. ย้อนหลังจากเวลาที่วัด (ไม่ใช่ฝนตามวันปฏิทิน), rain_3hr(mm) = ฝนช่วง 3 ชม.ล่าสุด
#   - แต่ละสถานี: วันที่วัด = วันนี้ -> today | = เมื่อวาน -> yesterday | เก่ากว่านั้น -> ข้าม (สถานีไม่ส่งข้อมูล)
#   - ตัวเลขเก็บเป็นข้อความตามที่ API ส่งมา (API ส่งเป็น string อยู่แล้ว) + เก็บ JSON ดิบไว้ใน *_raw.json
#   - วิธีใช้: รัน cell 1 (import) และ cell 2 แล้วรัน cell นี้ ไม่ต้องรัน cell ดึงรายปี
today_str = datetime.now().strftime("%Y-%m-%d")
yesterday_str = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

rain_today_api_url = "https://data.tmd.go.th/api/Weather3Hours/V2/?uid=api&ukey=api12345&format=json"
os.makedirs("tmd_today", exist_ok=True)
# [note 13] ใส่เวลาที่รันในชื่อไฟล์ด้วย -> ดึงวันเดียวกันคนละเวลาได้ไฟล์แยกกัน ไม่ทับกัน
# today_file = "tmd_today/" + today_str + ".csv"   # old
# raw_file = "tmd_today/" + today_str + "_raw.json"   # old
now_str = datetime.now().strftime("%Y-%m-%d_%H%M")   # new
today_file = "tmd_today/" + now_str + ".csv"   # new
raw_file = "tmd_today/" + now_str + "_raw.json"   # new

def text_or_blank(v):
    return v if isinstance(v, str) else ""   # API ส่งช่องที่ไม่มีค่าเป็น {}

response = requests.get(rain_today_api_url, timeout=60)
print("response status: %d" % response.status_code)
response.raise_for_status()   # [note 12] new: status ไม่ใช่ 200 -> raise ให้ task fail (เดิมจบเงียบ ๆ)
if (response.status_code == 200):
    data_list = response.json()["Stations"]["Station"]
    with open(raw_file, 'w', encoding="utf-8") as file:
        file.write(response.text)
    n_today = 0
    n_yesterday = 0
    n_skip = 0
    with open(today_file, 'w', encoding="tis-620", errors="replace", newline='') as file:
        writer = csv.writer(file)
        header_row = ["station_id","station_name_th","station_name_en","province","lat","lon","date","datetime","rain(mm)","rain_3hr(mm)","temp(c)","windDirection","windSpeed(km/h)","data_from"]
        writer.writerow(header_row)
        for dd in data_list:
            obs = dd["Observation"]
            obs_datetime = datetime.strptime(obs["DateTime"], "%m/%d/%Y %H:%M:%S")   # API ส่งมาแบบ 09/30/2026 04:00:00
            data_date = obs_datetime.strftime("%Y-%m-%d")
            if (data_date == today_str):
                data_from = "today"
                n_today = n_today + 1
            elif (data_date == yesterday_str):
                data_from = "yesterday"
                n_yesterday = n_yesterday + 1
            else:
                n_skip = n_skip + 1
                continue
            drow = [dd["WmoStationNumber"],text_or_blank(dd["StationNameThai"]),text_or_blank(dd["StationNameEnglish"]),text_or_blank(dd["Province"]),dd["Latitude"],dd["Longitude"],data_date,obs_datetime.strftime("%Y-%m-%d %H:%M:%S"),text_or_blank(obs.get("Rainfall24Hr")),text_or_blank(obs.get("Rainfall")),text_or_blank(obs.get("AirTemperature")),text_or_blank(obs.get("WindDirection")),text_or_blank(obs.get("WindSpeed")),data_from]
            writer.writerow(drow)
    print("today (%s): %d stations" % (today_str, n_today))
    print("yesterday (%s): %d stations" % (yesterday_str, n_yesterday))
    print("skip (older data): %d stations" % n_skip)
    print("saved:", today_file)

response status: 200
today (2026-09-30): 117 stations
yesterday (2026-09-29): 0 stations
skip (older data): 0 stations
saved: tmd_today/2026-09-30.csv
